# Download THINGS macaque / human data

This notebook downloads and prepares the publicly available data needed to run this project.

Use the `primate_align` environment and launch Jupyter from this repository. The default data location comes from `config/config.toml`. Please set `PRIMATE_ALIGN_DIR` **before starting Jupyter**; the same variable must be present when running the downstream analyses.

The final time-resolved section is optional if you want the underlying data for Supplementary Figure 1 or use it yourself. It downloads roughly 2 × 60 GB and also requires substantial working memory.


## External visual ratings

No additional download is needed for the analyses: the exact values used in the paper are included in `data/features/vis_props/vis_props.tsv`.

Real-world size is the concept-level `size_mean` rating from THINGSplus, joined to images by `uniqueID`. The source table is `02_object-level/_property-ratings.tsv` in the [official THINGS OSF repository](https://osf.io/jum2f/files/osfstorage). Curvature comes from a separate set of image-level human ratings and is not part of THINGSplus; the values required here are included in the bundled visual feature table.


In [ ]:
import os
import sys
import pickle
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# Locate the repository from either the repo root or fetch/ directory.
search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in search_roots if (p / 'config' / 'config.toml').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate the repository. Launch Jupyter from the repository root or its fetch directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.paths import config

REQUEST_TIMEOUT = (30, 300)
CHUNK_SIZE = 1024 * 1024
DOWNLOAD_TIME_RESOLVED = False

def log_msg(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

log_msg(f'Project root: {PROJECT_ROOT}')
log_msg(f'Data root: {config.data_dir}')
if os.environ.get('PRIMATE_ALIGN_DIR'):
    log_msg('Using PRIMATE_ALIGN_DIR override')

In [ ]:
def _nonempty_directory(path):
    path = Path(path)
    return path.is_dir() and any(p.name != '.download_complete' for p in path.iterdir())


def download_file(url, filepath):
    """Stream a file to an atomic temporary path and reuse completed files."""
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    if filepath.is_file() and filepath.stat().st_size > 0:
        log_msg(f'Already exists: {filepath}')
        return filepath

    partial = filepath.with_name(filepath.name + '.part')
    partial.unlink(missing_ok=True)
    log_msg(f'Downloading {filepath.name}...')
    try:
        with requests.get(url, stream=True, timeout=REQUEST_TIMEOUT) as response:
            response.raise_for_status()
            total = int(response.headers.get('content-length', 0))
            with open(partial, 'wb') as handle, tqdm(
                total=total or None, unit='B', unit_scale=True, desc=filepath.name
            ) as progress:
                for chunk in response.iter_content(CHUNK_SIZE):
                    if chunk:
                        handle.write(chunk)
                        progress.update(len(chunk))
        if partial.stat().st_size == 0:
            raise IOError(f'Empty download: {url}')
        partial.replace(filepath)
    except Exception:
        partial.unlink(missing_ok=True)
        raise
    log_msg(f'Saved: {filepath}')
    return filepath


def _extract_checked(archive, staging, password=None):
    """Validate and extract a ZIP without allowing paths outside staging."""
    staging = Path(staging)
    staging.mkdir(parents=True, exist_ok=True)
    root = staging.resolve()
    with zipfile.ZipFile(archive, 'r') as zf:
        pwd = password.encode() if password else None
        if pwd:
            zf.setpassword(pwd)
        for member in zf.infolist():
            target = (staging / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f'Unsafe archive member: {member.filename}')
        bad_member = zf.testzip()
        if bad_member is not None:
            raise zipfile.BadZipFile(f'CRC failure in {bad_member}')
        zf.extractall(staging, pwd=pwd)


def _merge_tree(source, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for item in Path(source).iterdir():
        target = destination / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)


def download_zip(url, destination, *, ready_path=None, password=None, flatten_folder=None):
    """Download into staging, validate, then merge the complete extraction."""
    destination = Path(destination)
    ready_path = Path(ready_path) if ready_path is not None else destination
    marker = ready_path / '.download_complete'
    in_progress = ready_path.parent / f'.{ready_path.name}.download_in_progress'
    if in_progress.exists():
        log_msg(f'Removing an incomplete extraction: {ready_path}')
        if ready_path.is_dir():
            shutil.rmtree(ready_path)
        else:
            ready_path.unlink(missing_ok=True)
        in_progress.unlink()
    if marker.exists() or _nonempty_directory(ready_path):
        log_msg(f'Already exists: {ready_path}')
        return destination

    destination.parent.mkdir(parents=True, exist_ok=True)
    ready_path.parent.mkdir(parents=True, exist_ok=True)
    token = ready_path.name.replace(' ', '_')
    archive = destination.parent / f'.{token}.download.zip'
    staging = destination.parent / f'.{token}.extracting'
    shutil.rmtree(staging, ignore_errors=True)
    archive.unlink(missing_ok=True)
    in_progress.write_text(f'{url}\n', encoding='utf-8')
    try:
        download_file(url, archive)
        log_msg(f'Extracting {ready_path.name}...')
        _extract_checked(archive, staging, password=password)
        source = staging / flatten_folder if flatten_folder else staging
        if flatten_folder and not source.is_dir():
            raise FileNotFoundError(f'Expected {flatten_folder!r} inside archive')
        if not _nonempty_directory(source):
            raise IOError(f'Archive extracted no files: {url}')
        _merge_tree(source, destination)
        ready_path.mkdir(parents=True, exist_ok=True)
        marker.write_text(f'{url}\n', encoding='utf-8')
        in_progress.unlink()
    finally:
        archive.unlink(missing_ok=True)
        shutil.rmtree(staging, ignore_errors=True)
    log_msg(f'Completed: {ready_path}')
    return destination


def atomic_to_csv(frame, filepath):
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    partial = filepath.with_name(filepath.name + '.part')
    frame.to_csv(partial, index=False)
    partial.replace(filepath)


def atomic_save_numpy(value, filepath):
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    partial = filepath.with_name(filepath.name + '.part')
    with open(partial, 'wb') as handle:
        np.save(handle, value)
    partial.replace(filepath)


def atomic_pickle(value, filepath):
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    partial = filepath.with_name(filepath.name + '.part')
    with open(partial, 'wb') as handle:
        pickle.dump(value, handle, protocol=4)
    partial.replace(filepath)

In [ ]:
# Create only the stable directory skeleton.
for directory in [config.data_dir, config.things_dir, config.macq_dir, config.timeavg_dir, config.timeres_dir, config.mri_dir]:
    Path(directory).mkdir(parents=True, exist_ok=True)
log_msg('Directory structure ready')

## THINGS metadata and behavioral features

In [ ]:
things_prop_path = Path(config.things_dir) / 'things_property.tsv'
things_meta_path = Path(config.things_dir) / 'things_metadata.tsv'
download_file('https://osf.io/download/7cz69/', things_prop_path)
download_file('https://osf.io/download/um6a9/', things_meta_path)

log_msg('Processing THINGS properties...')
prop_df = pd.read_csv(things_prop_path, sep='\t')
meta_df = pd.read_csv(things_meta_path, sep='\t')
mean_cols = [c for c in prop_df.columns if c.startswith('property_') and c.endswith('_mean')]
prop_simple = prop_df[['uniqueID', *mean_cols]].copy()
prop_simple.columns = [
    c.replace('property_', '').replace('_mean', '') if c != 'uniqueID' else c
    for c in prop_simple.columns
]
prop_simple['uniqueID'] = prop_simple['uniqueID'].astype(str)
meta_df['Word'] = meta_df['Word'].astype(str)
things_props = (
    prop_simple
    .merge(meta_df[['Word', 'All Bottom-up Categories']], left_on='uniqueID', right_on='Word', how='left')
    .drop(columns='Word')
    .rename(columns={'All Bottom-up Categories': 'categories'})
)
atomic_to_csv(things_props, Path(config.things_dir) / 'things_properties.csv')
log_msg(f'Processed {len(things_props)} THINGS concepts')

In [ ]:
download_file(
    'https://raw.githubusercontent.com/ViCCo-Group/dimension_encoding/master/data/Categories_final_20200131_fixedUniqueID.tsv',
    Path(config.things_dir) / 'Categories_final_20200131_fixedUniqueID.tsv',
)

level_urls = [
    ('https://files.osf.io/v1/resources/jum2f/providers/osfstorage/66d068fc9e146696878c987e/?zip=', '01_image-level'),
    ('https://files.osf.io/v1/resources/jum2f/providers/osfstorage/66d068c4f6f9282a0989c1e1/?zip=', '02_object-level'),
    ('https://files.osf.io/v1/resources/jum2f/providers/osfstorage/66d06922c27bea2c64180627/?zip=', '03_category-level'),
]
for url, folder in level_urls:
    download_zip(url, Path(config.things_dir) / folder)

In [ ]:
download_zip(
    'https://files.osf.io/v1/resources/f5rn6/providers/osfstorage/62d6b2b5f66a943a782311a1/?zip=',
    Path(config.things_dir) / 'behav_embed' / 'data',
)
download_zip(
    'https://files.osf.io/v1/resources/f5rn6/providers/osfstorage/62d6b2edf66a943a71230223/?zip=',
    Path(config.things_dir) / 'behav_embed' / 'variables',
)

## THINGS object images

The OSF image archive is password-protected; get the password from the [THINGS OSF project](https://osf.io/jum2f/) and paste it below.


In [ ]:
log_msg('Preparing THINGS images; this is a large download')
things_img_password = ''  # paste THINGS image-archive password here
download_zip(
    'https://osf.io/download/rdxy2/',
    Path(config.things_dir) / 'images',
    password=things_img_password or None,
    flatten_folder='object_images',
)


## Macaque time-averaged multi-unit activity

Data courtesy of Paolo Papale and the Roelfsema group. The processed dictionary keys and matrix orientations below are the interface consumed by `functions/load.py`.

In [ ]:
macaque_urls = {
    'F': {
        'data': 'https://gin.g-node.org/paolo_papale/TVSD/raw/master/monkeyF/THINGS_normMUA.mat',
        'imgs': 'https://gin.g-node.org/paolo_papale/TVSD/raw/master/monkeyF/_logs/things_imgs.mat',
    },
    'N': {
        'data': 'https://gin.g-node.org/paolo_papale/TVSD/raw/master/monkeyN/THINGS_normMUA.mat',
        'imgs': 'https://gin.g-node.org/paolo_papale/TVSD/raw/master/monkeyN/_logs/things_imgs.mat',
    },
}
data_keys = ['SNR', 'SNR_max', 'lats', 'reliab', 'oracle', 'train_MUA', 'test_MUA', 'test_MUA_reps', 'tb']
img_keys = ['class', 'things_path']

for monkey, urls in macaque_urls.items():
    paths = {key: Path(value) for key, value in config.get_macq_paths(monkey).items()}
    data_mat = Path(config.timeavg_dir) / f'monkey{monkey}_THINGS_normMUA.mat'
    img_mat = Path(config.timeavg_dir) / f'monkey{monkey}_things_imgs.mat'

    if paths['data'].is_file() and paths['data'].stat().st_size > 0:
        log_msg(f'Monkey {monkey}: processed neural data already exist')
    else:
        download_file(urls['data'], data_mat)
        download_file(urls['imgs'], img_mat)
        log_msg(f'Monkey {monkey}: converting MATLAB inputs')

        image_data = {}
        with h5py.File(img_mat, 'r') as handle:
            for split in ['train_imgs', 'test_imgs']:
                prefix = split.split('_')[0]
                n_items = handle[f'{split}/class'].shape[0]
                for key in img_keys:
                    dataset = handle[f'{split}/{key}']
                    strings = []
                    for i in range(n_items):
                        values = handle[dataset[i, 0]][()].flatten().astype(int)
                        strings.append(''.join(chr(c) for c in values if c < 128))
                    image_data[f'{prefix}_{key}'] = strings

        with h5py.File(data_mat, 'r') as handle:
            neural_data = {key: np.array(handle[key]).T for key in data_keys}
        neural_data.update(image_data)
        atomic_save_numpy(neural_data, paths['data'])
        data_mat.unlink(missing_ok=True)
        img_mat.unlink(missing_ok=True)

    if not paths['stiminfo'].exists():
        neural_data = np.load(paths['data'], allow_pickle=True).item()
        exemplars = [
            os.path.splitext(os.path.basename(path.replace('\\', '/')))[0]
            for path in neural_data['train_things_path']
        ]
        stim_df = pd.DataFrame({'category': neural_data['train_class'], 'exemplar': exemplars})
        atomic_to_csv(stim_df, paths['stiminfo'])
    log_msg(f"Monkey {monkey}: ready ({len(pd.read_csv(paths['stiminfo']))} training images)")

## Human fMRI data and metadata

In [ ]:
# These archives contain brainmasks/ and betas_csv/ at their top level.
download_zip(
    'https://ndownloader.figshare.com/files/36682242',
    Path(config.mri_dir),
    ready_path=Path(config.mri_dir) / 'brainmasks',
)
download_zip(
    'https://ndownloader.figshare.com/files/43635873',
    Path(config.mri_dir),
    ready_path=Path(config.mri_dir) / 'betas_csv',
)

In [ ]:
betas_dir = Path(config.mri_dir) / 'betas_csv'
for subject in ['01', '02', '03']:
    paths = {key: Path(value) for key, value in config.get_mri_paths(subject).items()}
    if not paths['stiminfo'].exists():
        metadata = pd.read_csv(betas_dir / f'sub-{subject}_StimulusMetadata.csv')
        exemplars = metadata['stimulus'].astype(str).map(lambda value: os.path.splitext(value)[0])
        categories = exemplars.map(lambda value: value.rsplit('_', 1)[0])
        atomic_to_csv(pd.DataFrame({'category': categories, 'exemplar': exemplars}), paths['stiminfo'])
    log_msg(f"Human {subject}: ready ({len(pd.read_csv(paths['stiminfo']))} images)")

## Validate the main inputs

This is intentionally lightweight: it verifies the filenames, dictionary keys, stimulus counts, and fMRI peripherals used by the downstream loaders without loading the large fMRI response matrices into memory.

In [ ]:
required_monkey_keys = {
    'SNR', 'SNR_max', 'lats', 'reliab', 'oracle', 'train_MUA', 'test_MUA',
    'test_MUA_reps', 'tb', 'train_class', 'test_class', 'train_things_path', 'test_things_path',
}
for monkey in ['N', 'F']:
    paths = {key: Path(value) for key, value in config.get_macq_paths(monkey).items()}
    values = np.load(paths['data'], allow_pickle=True).item()
    missing = required_monkey_keys.difference(values)
    assert not missing, f'Monkey {monkey} missing keys: {sorted(missing)}'
    metadata = pd.read_csv(paths['stiminfo'])
    assert list(metadata.columns) == ['category', 'exemplar']
    assert len(metadata) == values['train_MUA'].shape[1] == len(values['train_things_path'])
    log_msg(f"Monkey {monkey}: {values['train_MUA'].shape[1]} stimuli × {values['train_MUA'].shape[0]} channels")

for subject in ['01', '02', '03']:
    paths = {key: Path(value) for key, value in config.get_mri_paths(subject).items()}
    for key in ['betas', 'stiminfo', 'brainmask', 'voxmeta']:
        assert paths[key].exists(), f'Human {subject} missing {key}: {paths[key]}'
    metadata = pd.read_csv(paths['stiminfo'])
    assert list(metadata.columns) == ['category', 'exemplar']
    log_msg(f'Human {subject}: {len(metadata)} stimulus rows and all peripherals present')

assert (Path(config.things_dir) / 'things_properties.csv').exists()
assert Path(config.categories_tsv).exists()
log_msg('MAIN DOWNLOADS COMPLETE')
log_msg(f'Data saved to: {config.data_dir}')

## Noise-ceiling check (time-averaged macaque)

This preserves the original diagnostic plot for comparison with the source publication ([doi:10.1016/j.neuron.2024.12.003](https://doi.org/10.1016/j.neuron.2024.12.003)).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

def create_noise_ceiling_cmap():
    rgb_vals = np.array([
        [255, 255, 229], [255, 255, 212], [255, 247, 188], [254, 227, 145],
        [254, 217, 142], [254, 196, 79], [254, 153, 41], [236, 112, 20],
        [217, 95, 14], [204, 76, 2], [153, 52, 4], [140, 45, 4], [102, 37, 6],
    ]) / 255.0
    n_bins, vthresh, vmax = 256, 0.17, 0.95
    colors = np.zeros((n_bins, 4))
    threshold = int(n_bins * vthresh / vmax)
    colors[:threshold] = np.array([0.85, 0.85, 0.85, 1])
    colors[threshold:] = LinearSegmentedColormap.from_list('ylorbr', rgb_vals)(
        np.linspace(0, 1, n_bins - threshold)
    )
    return ListedColormap(colors)

n_chan, arr_shape = 64, (8, 8)
regions = {
    'N': {'v1': (0, 512), 'v4': (512, 768), 'it': (768, 1024)},
    'F': {'v1': (0, 512), 'it': (512, 832), 'v4': (832, 1024)},
}
oracles = {
    monkey: np.squeeze(np.load(config.get_macq_paths(monkey)['data'], allow_pickle=True).item()['oracle'])
    for monkey in ['N', 'F']
}

fig = plt.figure(figsize=(12, 8))
grid = plt.GridSpec(2, 3)
image = None
for monkey_index, (monkey, oracle) in enumerate(oracles.items()):
    arrays_by_region = {region: [] for region in ['v1', 'v4', 'it']}
    for region, (start, end) in regions[monkey].items():
        for array_index in range((end - start) // n_chan):
            values = oracle[start + array_index * n_chan:start + (array_index + 1) * n_chan]
            arrays_by_region[region].append(np.fliplr(values.reshape(arr_shape)))
    for region_index, region in enumerate(['v1', 'v4', 'it']):
        axis = fig.add_subplot(grid[monkey_index, region_index])
        arrays = arrays_by_region[region]
        if arrays:
            image = axis.imshow(np.vstack(arrays), cmap=create_noise_ceiling_cmap(), aspect='equal', vmin=0, vmax=0.95)
            for boundary in range(1, len(arrays)):
                axis.axhline(boundary * arr_shape[0] - 0.5, color='white', linewidth=1)
            if monkey_index == 0:
                axis.set_title(f'{region.upper()} (n={len(arrays)})', pad=10)
            if region_index == 0:
                axis.set_ylabel(f'Monkey {monkey}', fontsize=12)
        axis.set_xticks([])
        axis.set_yticks([])

color_axis = fig.add_axes([0.92, 0.15, 0.02, 0.7])
colorbar = plt.colorbar(image, cax=color_axis)
colorbar.set_label("Noise ceiling\n(Oracle Pearson's r)", fontsize=10)
colorbar.set_ticks([0.17, 0.95])
colorbar.set_ticklabels(['0.17', '0.95'])
plt.tight_layout()
plt.show()
log_msg('Noise-ceiling visualization complete')

## Optional: time-resolved macaque data (Supplementary Figure 1)

Time-resolved macaque downloads are disabled by default (`DOWNLOAD_TIME_RESOLVED = False`). Set this to `True` in the setup cell to download these data. Expect roughly 120 GB of downloads and high peak memory use during conversion.

In [ ]:
trials_urls = {
    'F': 'https://gin.g-node.org/paolo_papale/TVSD/raw/master/monkeyF/THINGS_MUA_trials.mat',
    'N': 'https://gin.g-node.org/paolo_papale/TVSD/raw/master/monkeyN/THINGS_MUA_trials.mat',
}

if not DOWNLOAD_TIME_RESOLVED:
    log_msg('Skipping optional time-resolved macaque data')
else:
    for monkey, url in trials_urls.items():
        paths = {key: Path(value) for key, value in config.get_macq_tr_paths(monkey).items()}
        mat_path = Path(config.timeres_dir) / f'monkey{monkey}_THINGS_MUA_trials.mat'
        if paths['data'].is_file() and paths['data'].stat().st_size > 0:
            log_msg(f'Monkey {monkey}: processed time-resolved data already exist')
        else:
            download_file(url, mat_path)
            log_msg(f'Monkey {monkey}: converting time-resolved data')
            with h5py.File(mat_path, 'r') as handle:
                allmua = np.array(handle['ALLMUA'])
                allmat = np.array(handle['ALLMAT'])
                timebase = np.array(handle['tb']).flatten()
            valid = allmat[1, :].astype(int) > 0
            trial_data = {'mua': allmua[:, valid, :], 'trial_info': allmat[:, valid], 'timepoints': timebase}
            atomic_pickle(trial_data, paths['data'])
            mat_path.unlink(missing_ok=True)

        if not paths['stiminfo'].exists():
            with open(paths['data'], 'rb') as handle:
                trial_data = pickle.load(handle)
            trial_info = trial_data['trial_info'][1, :].astype(int)
            valid = trial_info > 0
            stimulus_ids = trial_info[valid] - 1
            static = np.load(config.get_macq_paths(monkey)['data'], allow_pickle=True).item()
            categories = [static['train_class'][i] for i in stimulus_ids]
            exemplars = [
                os.path.splitext(os.path.basename(static['train_things_path'][i]))[0]
                for i in stimulus_ids
            ]
            atomic_to_csv(pd.DataFrame({
                'trial_idx': np.where(valid)[0],
                'stim_id': trial_info[valid],
                'category': categories,
                'exemplar': exemplars,
            }), paths['stiminfo'])
        log_msg(f"Monkey {monkey}: time-resolved data ready ({len(pd.read_csv(paths['stiminfo']))} trials)")